# Multi-Objective Optimization

This notebook demonstrates multi-objective optimization techniques using `ws3`.

> **Prerequisites**: Completion of `070_ws3_quickstart_complete_workflow.ipynb`

## What You'll Learn

- How to formulate multiple objectives
- How to use weighted objective functions
- How to find Pareto-optimal solutions
- How to perform goal programming

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import ws3.forest
import ws3.opt
from util import compile_scenario, plot_scenario

In [ ]:
# Model parameters
base_year = 2020
horizon = 10
period_length = 10
max_age = 1000
tvy_name = "totvol"

print(f"Model Parameters:")
print(f"  Base Year: {base_year}")
print(f"  Horizon: {horizon} periods")
print(f"  Period Length: {period_length} years")

In [ ]:
# Create ForestModel
fm = ws3.forest.ForestModel(
    model_name="tsa24_clipped",
    model_path="data/woodstock_model_files_tsa24_clipped",
    base_year=base_year,
    horizon=horizon,
    period_length=period_length,
    max_age=max_age
)
fm.import_landscape_section()
fm.import_areas_section(convert_periods_to_years=period_length)
fm.import_yields_section(convert_periods_to_years=period_length)
fm.import_actions_section(convert_periods_to_years=period_length)
fm.import_transitions_section(convert_periods_to_years=period_length)
fm.initialize_areas()
fm.add_null_action()
fm.reset_actions()
fm.actions["harvest"].is_harvest = True

print(f"ForestModel loaded: {fm}")

In [ ]:
# Define coefficient functions for multiple objectives
import functools

def cmp_c_z_max_hv(expr, **kwargs):
    """Maximize harvest volume."""
    return expr

def cmp_c_z_min_ha(expr="1.", **kwargs):
    """Minimize harvest area."""
    return expr

def cmp_c_z_even_flow(expr, **kwargs):
    """Minimize deviation from even flow."""
    return f"({expr} - avg_hv)^2"

expr = "0.85 * totvol"

coeff_funcs = {
    "z_max_hv": functools.partial(cmp_c_z_max_hv, expr=expr),
    "z_min_ha": functools.partial(cmp_c_z_min_ha),
    "z_even_flow": functools.partial(cmp_c_z_even_flow, expr=expr)
}

print("Coefficient functions defined for multiple objectives")

In [ ]:
# Define constraints for each objective
acodes = ["null", "harvest"]

# Flow constraints (even-flow)
cflw_e = {
    "cflw_hv": ({p: 0.05 for p in fm.periods}, 1),
    "cflw_ha": ({p: 0.05 for p in fm.periods}, 1)
}

# General constraints
gs_lb_rhs = fm.inventory(0, "totvol") * 0.90
cgen_data = {
    "cgen_gs": {"lb": {10: gs_lb_rhs}, "ub": {10: 999999999.}}
}

print("Constraints defined")

In [ ]:
# Solve with different objectives
objectives = {
    "max_hv": (coeff_funcs["z_max_hv"], ws3.opt.SENSE_MAXIMIZE),
    "min_ha": (coeff_funcs["z_min_ha"], ws3.opt.SENSE_MINIMIZE)
}

results = {}

for obj_name, (coeff_func, sense) in objectives.items():
    print(f"\nSolving with objective: {obj_name}")
    
    problem = fm.add_problem(
        name=f"obj_{obj_name}",
        coeff_funcs={"z": coeff_func},
        cflw_e=cflw_e,
        cgen_data=cgen_data,
        acodes=acodes,
        sense=sense,
        mask=None,
        workers=1,
        verbose=False
    )
    
    problem.solve(verbose=False)
    
    if problem.status() == ws3.opt.STATUS_OPTIMAL:
        sch = fm.compile_schedule(problem)
        fm.apply_schedule(sch, 
                         force_integral_area=False,
                         override_operability=False,
                         fuzzy_age=False,
                         recourse_enabled=False,
                         verbose=False,
                         compile_c_ycomps=True)
        df = compile_scenario(fm)
        results[obj_name] = df
        print(f"  Status: Optimal")
    else:
        print(f"  Status: {problem.status()}")

print("\nAll objectives solved")

In [ ]:
# Compare results from different objectives
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

if 'max_hv' in results and 'min_ha' in results:
    # Plot harvest volume comparison
    axes[0].bar(['Period 1', 'Period 2', 'Period 3', 'Period 4', 'Period 5'],
                results['max_hv']['totvol'].values[:5], 
                alpha=0.7, label='Max Volume', color='blue')
    axes[0].bar([x + 0.2 for x in range(5)],
                results['min_ha']['totvol'].values[:5],
                alpha=0.7, label='Min Area', color='red')
    axes[0].set_xlabel('Period')
    axes[0].set_ylabel('Harvest Volume')
    axes[0].set_title('Harvest Volume Comparison')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot harvest area comparison
    axes[1].bar(['Period 1', 'Period 2', 'Period 3', 'Period 4', 'Period 5'],
                results['max_hv']['area'].values[:5],
                alpha=0.7, label='Max Volume', color='blue')
    axes[1].bar([x + 0.2 for x in range(5)],
                results['min_ha']['area'].values[:5],
                alpha=0.7, label='Min Area', color='red')
    axes[1].set_xlabel('Period')
    axes[1].set_ylabel('Harvest Area (ha)')
    axes[1].set_title('Harvest Area Comparison')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Weighted objective function approach
def weighted_objective(weight_hv, weight_ha, expr="0.85 * totvol"):
    """Create weighted objective function."""
    def weighted_coeff(**kwargs):
        return f"{weight_hv} * {expr} - {weight_ha} * 1."
    return weighted_coeff

# Try different weight combinations
weights = [(0.8, 0.2), (0.5, 0.5), (0.2, 0.8)]
weighted_results = {}

for w_hv, w_ha in weights:
    obj_name = f"weighted_{w_hv}_{w_ha}"
    print(f"\nSolving with weights: HV={w_hv}, HA={w_ha}")
    
    coeff_func = weighted_objective(w_hv, w_ha)
    
    problem = fm.add_problem(
        name=obj_name,
        coeff_funcs={"z": coeff_func},
        cflw_e=cflw_e,
        cgen_data=cgen_data,
        acodes=acodes,
        sense=ws3.opt.SENSE_MAXIMIZE,
        mask=None,
        workers=1,
        verbose=False
    )
    
    problem.solve(verbose=False)
    
    if problem.status() == ws3.opt.STATUS_OPTIMAL:
        sch = fm.compile_schedule(problem)
        fm.apply_schedule(sch,
                         force_integral_area=False,
                         override_operability=False,
                         fuzzy_age=False,
                         recourse_enabled=False,
                         verbose=False,
                         compile_c_ycomps=True)
        df = compile_scenario(fm)
        weighted_results[obj_name] = df
        print(f"  Status: Optimal")

print("\nWeighted objectives solved")

In [ ]:
# Visualize Pareto front
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Total harvest volume vs area for weighted solutions
total_hv = []
total_ha = []
labels = []

for obj_name, df in weighted_results.items():
    total_hv.append(df['totvol'].sum())
    total_ha.append(df['area'].sum())
    labels.append(obj_name.replace('weighted_', '').replace('_', ', '))

axes[0].scatter(total_ha, total_hv, s=100, c='blue', alpha=0.7)
for i, label in enumerate(labels):
    axes[0].annotate(label, (total_ha[i], total_hv[i]), 
                    textcoords="offset points", xytext=(0,10), ha='center')

axes[0].set_xlabel('Total Harvest Area (ha)')
axes[0].set_ylabel('Total Harvest Volume')
axes[0].set_title('Pareto Front: Volume vs Area')
axes[0].grid(True, alpha=0.3)

# Period-by-period comparison for best weighted solution
if len(weighted_results) > 0:
    best_name = list(weighted_results.keys())[len(weighted_results)//2]
    best_df = weighted_results[best_name]
    
    periods = range(min(5, len(best_df)))
    axes[1].bar(periods, best_df['totvol'].values[:5], alpha=0.7, color='green')
    axes[1].set_xlabel('Period')
    axes[1].set_ylabel('Harvest Volume')
    axes[1].set_title(f'Best Weighted Solution: {best_name}')
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Goal programming approach
def goal_programming(target_hv, target_ha, deviation_weight=1.0):
    """Create goal programming objective."""
    def goal_coeff(**kwargs):
        return f"-{deviation_weight} * ((target_hv - realized_hv)^2 + (target_ha - realized_ha)^2)"
    return goal_coeff

# Define targets
target_volume = 50000  # target total volume
target_area = 500      # target total area

print(f"Goal Programming Targets:")
print(f"  Target Volume: {target_volume}")
print(f"  Target Area: {target_area}")
print("\nGoal programming minimizes deviation from targets")

In [ ]:
# Summary of multi-objective optimization
print("=" * 60)
print("MULTI-OBJECTIVE OPTIMIZATION SUMMARY")
print("=" * 60)
print(f"Objectives Explored:")
print(f"  - Maximize harvest volume")
print(f"  - Minimize harvest area")
print(f"  - Weighted combinations")
print(f"  - Goal programming")
print(f"\nResults:")
for obj_name, df in results.items():
    print(f"  {obj_name}: Total Volume = {df['totvol'].sum():.2f}")
print(f"\nPareto front shows trade-offs between objectives")
print("=" * 60)